# Notebook 02-CIC v2 — CIC-IDS2017 Model Training

**Methodology changes vs v1:**

1. **M6 fix**: trained on the 60% train slice from Notebook 01-CIC v2 (~120,018 samples), not 80%. The 20% calibration slice (~40,006 samples) is reserved for calibrator fitting in Notebook 03-CIC v2. The 20% test slice (~40,007 samples) is untouched until evaluation.

2. **RF max_depth=20 cap**: applied preemptively for consistency with UNSW v2. CIC didn't have the deep-tree problem in v1, but capping at 20 ensures all three datasets use the same RF configuration and removes any latent SHAP-infeasibility risk.

3. **Predictions on both test and calibration sets**: every model writes 4 prediction files (test_pred, test_proba, calib_pred, calib_proba). Notebook 03-CIC v2 will use calib_proba files to fit isotonic calibrators.

**Models trained (9 total):**

Canonical 6 (used by all downstream notebooks):
- `rf_binary_cw`, `xgb_binary_cw`, `dnn_binary_cw` — binary, class-weighted
- `rf_5class_smote`, `xgb_5class_smote`, `dnn_5class_smote` — 5-class, SMOTE

Ablation 3 (justifies SMOTE choice for 5-class):
- `rf_5class_cw`, `xgb_5class_cw`, `dnn_5class_cw` — 5-class, class-weighted

**Note on U2R class on CIC:** only 22 train / 7 calib / 7 test samples. Per-class F1 on U2R will have very high variance. Reported but flagged as low-confidence.

**Outputs written to `models/cic_ids2017_v2/`** — parallel to v1.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull origin main
print(f'\n✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
From https://github.com/anasbiswas1/xids-research
 * branch            main       -> FETCH_HEAD
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU. Switch to T4: Runtime → Change runtime type → T4 GPU.')

PyTorch device: cuda
GPU: Tesla T4


## 2. Load v2 preprocessed data

In [3]:
import numpy as np
import pandas as pd
import json, joblib, pickle, time
from pathlib import Path
from collections import Counter

REPO_P = Path(REPO)
PROC_DIR = REPO_P / 'data/processed/cic_ids2017_v2'
MODELS_DIR = REPO_P / 'models/cic_ids2017_v2'
PREDS_DIR = MODELS_DIR / 'predictions'
PROBS_DIR = MODELS_DIR / 'probabilities'
TABLES_DIR = REPO_P / 'results/tables'
for d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_calib = np.load(PROC_DIR / 'X_calib.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')

y_train_binary = np.load(PROC_DIR / 'y_train_binary.npy')
y_calib_binary = np.load(PROC_DIR / 'y_calib_binary.npy')
y_test_binary  = np.load(PROC_DIR / 'y_test_binary.npy')

y_train_5class = np.load(PROC_DIR / 'y_train_5class.npy')
y_calib_5class = np.load(PROC_DIR / 'y_calib_5class.npy')
y_test_5class  = np.load(PROC_DIR / 'y_test_5class.npy')

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)
with open(PROC_DIR / 'class_mappings.json') as f:
    class_mappings = json.load(f)

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
N_FEATURES = X_train.shape[1]

print(f'Train: {X_train.shape}  (60% slice)')
print(f'Calib: {X_calib.shape}  (20% slice — fed to Notebook 03 v2)')
print(f'Test:  {X_test.shape}   (20% slice)')
print(f'\nTrain 5-class distribution:')
for cid, name in enumerate(FIVE_CLASS_NAMES):
    n = int((y_train_5class == cid).sum())
    print(f'  {cid} {name:8s}: {n:>7,}')

Train: (120018, 78)  (60% slice)
Calib: (40006, 78)  (20% slice — fed to Notebook 03 v2)
Test:  (40007, 78)   (20% slice)

Train 5-class distribution:
  0 Normal  :  96,456
  1 DoS     :  16,126
  2 Probe   :   6,743
  3 R2L     :     671
  4 U2R     :      22


## 3. Evaluation and save helpers

In [4]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, roc_auc_score
)

ALL_METRICS = {}

def evaluate(name, y_true, y_pred, y_proba, target_type):
    metrics = {
        'model': name,
        'target_type': target_type,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
    }
    if target_type == 'binary':
        pos_proba = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
        metrics['auc_roc'] = float(roc_auc_score(y_true, pos_proba))
        metrics['per_class_f1'] = {
            'Normal': float(f1_score(y_true, y_pred, pos_label=0)),
            'Attack': float(f1_score(y_true, y_pred, pos_label=1)),
        }
    else:
        per_class = f1_score(y_true, y_pred, average=None, labels=range(5), zero_division=0)
        metrics['per_class_f1'] = {n: float(per_class[i]) for i, n in enumerate(FIVE_CLASS_NAMES)}
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred).tolist()
    return metrics

def save_artifacts(name, model, model_kind, predict_fn):
    if model_kind in ('sklearn', 'xgboost'):
        joblib.dump(model, MODELS_DIR / f'{name}.joblib')
    elif model_kind == 'torch':
        torch.save(model.state_dict(), MODELS_DIR / f'{name}.pt')
    test_pred, test_proba = predict_fn(X_test)
    calib_pred, calib_proba = predict_fn(X_calib)
    np.save(PREDS_DIR / f'{name}_test_pred.npy',   test_pred)
    np.save(PROBS_DIR / f'{name}_test_proba.npy',  test_proba)
    np.save(PREDS_DIR / f'{name}_calib_pred.npy',  calib_pred)
    np.save(PROBS_DIR / f'{name}_calib_proba.npy', calib_proba)
    return test_pred, test_proba

def checkpoint_metrics():
    with open(MODELS_DIR / 'metrics.json', 'w') as f:
        json.dump(ALL_METRICS, f, indent=2, default=str)

print('Helpers ready.')

Helpers ready.


## 4. RF binary, class-weighted (max_depth=20 cap)

In [5]:
from sklearn.ensemble import RandomForestClassifier

RF_MAX_DEPTH = 20

name = 'rf_binary_cw'
print(f'Training {name}...')
t0 = time.time()

clf = RandomForestClassifier(
    n_estimators=200, max_depth=RF_MAX_DEPTH,
    class_weight='balanced', n_jobs=-1, random_state=42,
)
clf.fit(X_train, y_train_binary)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'sklearn', predict_fn)
metrics = evaluate(name, y_test_binary, test_pred, test_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
metrics['max_depth_actual'] = int(max(t.tree_.max_depth for t in clf.estimators_))
metrics['max_depth_mean'] = float(np.mean([t.tree_.max_depth for t in clf.estimators_]))
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Tree depths: mean={metrics['max_depth_mean']:.1f}, max={metrics['max_depth_actual']} (cap was {RF_MAX_DEPTH})")
print(f"  Time: {metrics['train_time_sec']}s")

Training rf_binary_cw...
  Acc=0.9985  F1m=0.9976  MCC=0.9952  AUC=0.9998
  Tree depths: mean=20.0, max=20 (cap was 20)
  Time: 44.9s


## 5. XGB binary, class-weighted

In [6]:
import xgboost as xgb

name = 'xgb_binary_cw'
print(f'Training {name}...')
t0 = time.time()

spw = float((y_train_binary == 0).sum()) / float((y_train_binary == 1).sum())
print(f'  scale_pos_weight = {spw:.3f}')

clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    tree_method='hist', device='cuda' if device.type == 'cuda' else 'cpu',
    scale_pos_weight=spw, eval_metric='logloss',
    random_state=42, n_jobs=-1,
)
clf.fit(X_train, y_train_binary)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'xgboost', predict_fn)
metrics = evaluate(name, y_test_binary, test_pred, test_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Time: {metrics['train_time_sec']}s")

Training xgb_binary_cw...
  scale_pos_weight = 4.094


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [01:05:34] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  Acc=0.9990  F1m=0.9984  MCC=0.9968  AUC=0.9999
  Time: 3.3s


## 6. DNN architecture and training helper

In [7]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

class XIDSMLP(nn.Module):
    """256-128-64-32 MLP with BatchNorm, ReLU, dropout 0.3."""
    def __init__(self, n_features, n_classes, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(64, 32),          nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        return self.net(x)

def train_dnn(X_tr, y_tr, n_classes, class_weights=None,
              epochs=40, batch_size=512, lr=1e-3, patience=5, verbose=True):
    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_tr, y_tr, test_size=0.1, random_state=42, stratify=y_tr,
    )
    model = XIDSMLP(X_tr2.shape[1], n_classes).to(device)

    if class_weights is not None:
        cw_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw_t)
    else:
        criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    Xtr = torch.tensor(X_tr2, dtype=torch.float32)
    ytr = torch.tensor(y_tr2, dtype=torch.long)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    Xv = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv = torch.tensor(y_val, dtype=torch.long).to(device)

    best_val_loss = float('inf'); best_state = None; no_improve = 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_logits = model(Xv)
            val_loss = criterion(val_logits, yv).item()
            val_acc = (val_logits.argmax(1) == yv).float().mean().item()
        if verbose and (ep == 1 or ep % 5 == 0):
            print(f'  ep {ep:>3d}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose: print(f'  early stop at epoch {ep}')
                break
    model.load_state_dict(best_state)
    model.eval()
    return model

def dnn_predict(model):
    def predict_fn(X):
        Xte = torch.tensor(X, dtype=torch.float32).to(device)
        model.eval()
        with torch.no_grad():
            logits = model(Xte)
            proba = F.softmax(logits, dim=1).cpu().numpy()
            pred = proba.argmax(axis=1)
        return pred, proba
    return predict_fn

print('DNN helpers ready.')

DNN helpers ready.


## 7. DNN binary, class-weighted

In [8]:
from sklearn.utils.class_weight import compute_class_weight

name = 'dnn_binary_cw'
print(f'Training {name}...')
t0 = time.time()

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_binary)
print(f'  class weights: Normal={cw[0]:.3f}, Attack={cw[1]:.3f}')

model = train_dnn(X_train, y_train_binary, n_classes=2, class_weights=cw)
predict_fn = dnn_predict(model)
test_pred, test_proba = save_artifacts(name, model, 'torch', predict_fn)
metrics = evaluate(name, y_test_binary, test_pred, test_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Time: {metrics['train_time_sec']}s")

Training dnn_binary_cw...
  class weights: Normal=0.622, Attack=2.547
  ep   1  val_loss=0.1107  val_acc=0.9431
  ep   5  val_loss=0.0848  val_acc=0.9501
  ep  10  val_loss=0.0724  val_acc=0.9663
  ep  15  val_loss=0.0643  val_acc=0.9673
  early stop at epoch 19
  Acc=0.9694  F1m=0.9537  MCC=0.9101  AUC=0.9969
  Time: 31.7s


## 8. SMOTE on 5-class training data

Standard SMOTE (not modified) — oversample minority classes to match the largest class. CIC's class imbalance is smaller than UNSW's so standard SMOTE works cleanly. Special handling for U2R: only 22 training samples, so k_neighbors capped accordingly.

In [9]:
from imblearn.over_sampling import SMOTE

print('Pre-SMOTE 5-class distribution:')
pre = Counter(y_train_5class)
for cid in range(5):
    print(f'  {cid} {FIVE_CLASS_NAMES[cid]:8s}: {pre[cid]:>7,}')

# Compute safe k_neighbors based on minimum class size
min_class_size = min(pre.values())
k_neighbors = min(5, min_class_size - 1)
print(f'\nMin class size: {min_class_size}, using k_neighbors={k_neighbors}')

smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
X_train_sm, y_train_5class_sm = smote.fit_resample(X_train, y_train_5class)
X_train_sm = X_train_sm.astype(np.float32)

print('\nPost-SMOTE distribution:')
post = Counter(y_train_5class_sm)
for cid in range(5):
    print(f'  {cid} {FIVE_CLASS_NAMES[cid]:8s}: {post[cid]:>7,}')
print(f'\nResampled shape: {X_train_sm.shape}')

Pre-SMOTE 5-class distribution:
  0 Normal  :  96,456
  1 DoS     :  16,126
  2 Probe   :   6,743
  3 R2L     :     671
  4 U2R     :      22

Min class size: 22, using k_neighbors=5

Post-SMOTE distribution:
  0 Normal  :  96,456
  1 DoS     :  96,456
  2 Probe   :  96,456
  3 R2L     :  96,456
  4 U2R     :  96,456

Resampled shape: (482280, 78)


## 9. RF 5-class SMOTE (max_depth=20 cap)

In [10]:
name = 'rf_5class_smote'
print(f'Training {name}...')
t0 = time.time()

clf = RandomForestClassifier(
    n_estimators=200, max_depth=RF_MAX_DEPTH,
    n_jobs=-1, random_state=42,
)
clf.fit(X_train_sm, y_train_5class_sm)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'sklearn', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
metrics['max_depth_actual'] = int(max(t.tree_.max_depth for t in clf.estimators_))
metrics['max_depth_mean'] = float(np.mean([t.tree_.max_depth for t in clf.estimators_]))
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Tree depths: mean={metrics['max_depth_mean']:.1f}, max={metrics['max_depth_actual']} (cap was {RF_MAX_DEPTH})")
print(f"  Time: {metrics['train_time_sec']}s")

Training rf_5class_smote...
  Acc=0.9983  F1m=0.9571  MCC=0.9948
  Per-class F1: {'Normal': 0.9989421281891724, 'DoS': 0.9971213668864333, 'Probe': 0.9957787158409243, 'R2L': 0.9601769911504425, 'U2R': 0.8333333333333334}
  Tree depths: mean=20.0, max=20 (cap was 20)
  Time: 258.8s


## 10. XGB 5-class SMOTE

In [11]:
name = 'xgb_5class_smote'
print(f'Training {name}...')
t0 = time.time()

clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    tree_method='hist', device='cuda' if device.type == 'cuda' else 'cpu',
    objective='multi:softprob', num_class=5,
    eval_metric='mlogloss', random_state=42, n_jobs=-1,
)
clf.fit(X_train_sm, y_train_5class_sm)

def predict_fn(X):
    return clf.predict(X), clf.predict_proba(X)

test_pred, test_proba = save_artifacts(name, clf, 'xgboost', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Time: {metrics['train_time_sec']}s")

Training xgb_5class_smote...
  Acc=0.9989  F1m=0.9782  MCC=0.9966
  Per-class F1: {'Normal': 0.9992999050982466, 'DoS': 0.9979572887650882, 'Probe': 0.9973368841544608, 'R2L': 0.9732142857142857, 'U2R': 0.9230769230769231}
  Time: 19.0s


## 11. DNN 5-class SMOTE

In [12]:
name = 'dnn_5class_smote'
print(f'Training {name}...')
t0 = time.time()

cw = compute_class_weight('balanced', classes=np.arange(5), y=y_train_5class_sm)
print(f'  class weights: {dict(zip(FIVE_CLASS_NAMES, cw.round(3)))}')

model = train_dnn(X_train_sm, y_train_5class_sm, n_classes=5, class_weights=cw)
predict_fn = dnn_predict(model)
test_pred, test_proba = save_artifacts(name, model, 'torch', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()

print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Time: {metrics['train_time_sec']}s")

Training dnn_5class_smote...
  class weights: {'Normal': np.float64(1.0), 'DoS': np.float64(1.0), 'Probe': np.float64(1.0), 'R2L': np.float64(1.0), 'U2R': np.float64(1.0)}
  ep   1  val_loss=0.0712  val_acc=0.9774
  ep   5  val_loss=0.0436  val_acc=0.9856
  ep  10  val_loss=0.0404  val_acc=0.9858
  ep  15  val_loss=0.0367  val_acc=0.9872
  ep  20  val_loss=0.0368  val_acc=0.9880
  ep  25  val_loss=0.0367  val_acc=0.9861
  early stop at epoch 26
  Acc=0.9606  F1m=0.7265  MCC=0.8957
  Per-class F1: {'Normal': 0.9749358657722398, 'DoS': 0.9361367998604083, 'Probe': 0.9089805825242718, 'R2L': 0.5365853658536586, 'U2R': 0.27586206896551724}
  Time: 139.7s


## 12. Ablation: 5-class class-weighted variants

In [13]:
labels_5 = list(range(5))
cw_5 = compute_class_weight('balanced', classes=np.array(labels_5), y=y_train_5class)
cw_5_dict = {i: float(cw_5[i]) for i in labels_5}
sample_w_5 = np.array([cw_5[v] for v in y_train_5class])

print(f'5-class class weights:')
for i in range(5):
    print(f'  {FIVE_CLASS_NAMES[i]:8s}: {cw_5[i]:.4f}')

# RF 5-class CW
name = 'rf_5class_cw'
print(f'\nTraining {name}...')
t0 = time.time()
clf = RandomForestClassifier(n_estimators=200, max_depth=RF_MAX_DEPTH, class_weight=cw_5_dict, n_jobs=-1, random_state=42)
clf.fit(X_train, y_train_5class)
def predict_fn(X): return clf.predict(X), clf.predict_proba(X)
test_pred, test_proba = save_artifacts(name, clf, 'sklearn', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()
print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")

# XGB 5-class CW
name = 'xgb_5class_cw'
print(f'\nTraining {name}...')
t0 = time.time()
clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1, tree_method='hist',
    device='cuda' if device.type == 'cuda' else 'cpu',
    objective='multi:softprob', num_class=5, eval_metric='mlogloss',
    random_state=42, n_jobs=-1,
)
clf.fit(X_train, y_train_5class, sample_weight=sample_w_5)
def predict_fn(X): return clf.predict(X), clf.predict_proba(X)
test_pred, test_proba = save_artifacts(name, clf, 'xgboost', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()
print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")

# DNN 5-class CW
name = 'dnn_5class_cw'
print(f'\nTraining {name}...')
t0 = time.time()
model = train_dnn(X_train, y_train_5class, n_classes=5, class_weights=cw_5)
predict_fn = dnn_predict(model)
test_pred, test_proba = save_artifacts(name, model, 'torch', predict_fn)
metrics = evaluate(name, y_test_5class, test_pred, test_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
checkpoint_metrics()
print(f"  Acc={metrics['accuracy']:.4f}  F1m={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")

5-class class weights:
  Normal  : 0.2489
  DoS     : 1.4885
  Probe   : 3.5598
  R2L     : 35.7729
  U2R     : 1091.0727

Training rf_5class_cw...
  Acc=0.9982  F1m=0.9584  MCC=0.9946

Training xgb_5class_cw...
  Acc=0.9990  F1m=0.9799  MCC=0.9970

Training dnn_5class_cw...
  ep   1  val_loss=0.7652  val_acc=0.7800
  ep   5  val_loss=0.6389  val_acc=0.8811
  early stop at epoch 7
  Acc=0.8227  F1m=0.5400  MCC=0.6638


## 13. Summary table

In [14]:
rows = []
for name, m in ALL_METRICS.items():
    row = {
        'Model': name,
        'Target': m['target_type'],
        'Accuracy': round(m['accuracy'], 4),
        'F1-macro': round(m['f1_macro'], 4),
        'F1-weighted': round(m['f1_weighted'], 4),
        'MCC': round(m['mcc'], 4),
        'AUC-ROC': round(m['auc_roc'], 4) if m['target_type'] == 'binary' else None,
        'R2L F1': round(m['per_class_f1']['R2L'], 4) if 'R2L' in m['per_class_f1'] else None,
        'U2R F1': round(m['per_class_f1']['U2R'], 4) if 'U2R' in m['per_class_f1'] else None,
        'Time (s)': m['train_time_sec'],
    }
    rows.append(row)
df = pd.DataFrame(rows)
print('ALL 9 MODELS — CIC-IDS2017 v2')
print('=' * 110)
print(df.to_string(index=False))
print('=' * 110)
df.to_csv(TABLES_DIR / 'cic_v2_model_comparison.csv', index=False)
print(f'\nSaved: {TABLES_DIR / "cic_v2_model_comparison.csv"}')

ALL 9 MODELS — CIC-IDS2017 v2
           Model     Target  Accuracy  F1-macro  F1-weighted    MCC  AUC-ROC  R2L F1  U2R F1  Time (s)
    rf_binary_cw     binary    0.9985    0.9976       0.9985 0.9952   0.9998     NaN     NaN      44.9
   xgb_binary_cw     binary    0.9990    0.9984       0.9990 0.9968   0.9999     NaN     NaN       3.3
   dnn_binary_cw     binary    0.9694    0.9537       0.9701 0.9101   0.9969     NaN     NaN      31.7
 rf_5class_smote multiclass    0.9983    0.9571       0.9983 0.9948      NaN  0.9602  0.8333     258.8
xgb_5class_smote multiclass    0.9989    0.9782       0.9989 0.9966      NaN  0.9732  0.9231      19.0
dnn_5class_smote multiclass    0.9606    0.7265       0.9635 0.8957      NaN  0.5366  0.2759     139.7
    rf_5class_cw multiclass    0.9982    0.9584       0.9982 0.9946      NaN  0.9680  0.8333      46.2
   xgb_5class_cw multiclass    0.9990    0.9799       0.9990 0.9970      NaN  0.9709  0.9333       5.7
   dnn_5class_cw multiclass    0.8227    0.

## 14. Imbalance-strategy ablation table

In [15]:
ablation_rows = []
for arch in ['rf', 'xgb', 'dnn']:
    for strat in ['cw', 'smote']:
        name = f'{arch}_5class_{strat}'
        m = ALL_METRICS[name]
        ablation_rows.append({
            'Architecture': arch.upper(),
            'Strategy': 'class-weighted' if strat == 'cw' else 'SMOTE',
            'Accuracy': round(m['accuracy'], 4),
            'Macro F1': round(m['f1_macro'], 4),
            'MCC': round(m['mcc'], 4),
            'R2L F1': round(m['per_class_f1']['R2L'], 4),
            'U2R F1': round(m['per_class_f1']['U2R'], 4),
        })
df = pd.DataFrame(ablation_rows)
print('5-CLASS IMBALANCE STRATEGY ABLATION — CIC-IDS2017 v2')
print('=' * 100)
print(df.to_string(index=False))
print('=' * 100)
df.to_csv(TABLES_DIR / 'cic_v2_imbalance_ablation.csv', index=False)
print(f'\nSaved: {TABLES_DIR / "cic_v2_imbalance_ablation.csv"}')

5-CLASS IMBALANCE STRATEGY ABLATION — CIC-IDS2017 v2
Architecture       Strategy  Accuracy  Macro F1    MCC  R2L F1  U2R F1
          RF class-weighted    0.9982    0.9584 0.9946  0.9680  0.8333
          RF          SMOTE    0.9983    0.9571 0.9948  0.9602  0.8333
         XGB class-weighted    0.9990    0.9799 0.9970  0.9709  0.9333
         XGB          SMOTE    0.9989    0.9782 0.9966  0.9732  0.9231
         DNN class-weighted    0.8227    0.5400 0.6638  0.1003  0.0450
         DNN          SMOTE    0.9606    0.7265 0.8957  0.5366  0.2759

Saved: /content/drive/MyDrive/XIDS_Research/xids-research/results/tables/cic_v2_imbalance_ablation.csv


## 15. Commit and push

In [16]:
os.chdir(REPO)
!git add notebooks/02_cic_train_models_v2.ipynb
!git add results/tables/cic_v2_model_comparison.csv
!git add results/tables/cic_v2_imbalance_ablation.csv
!git status --short
!git commit -m 'Notebook 02-CIC v2: train 9 models on 60% slice with M6 fix + RF max_depth=20 cap'
!git push origin main

Refresh index: 100% (233/233), done.
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
AM notebooks/02_cic_train_models_v2.ipynb
 M notebooks/02_train_models_v2.ipynb
 M notebooks/02_unsw_train_models_v2.ipynb
A  results/tables/cic_v2_imbalance_ablation.csv
A  results/tables/cic_v2_model_comparison.csv
?? calibrators/
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? results/figures/unsw_dnn_5class_diagnostic.png
?? shap_values/unsw_nb15/
[main 0e35f0d] Notebook 02-CIC v2: train 9 models on 60% slice with M6 fix + RF max_depth=20 cap
 3 files changed, 18 insertions(+)
 create mode 100644 notebooks/02_cic_train_models_v2.ipynb
 create mode 100644 results/tables/cic_v2_imbalance_ablation.csv
 create mode 100644 results/tables/cic_v2_model_comparison.csv
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done